# V3 Raw + corrected structural mapping: GPT-3.5 Mixed → HC3 0%

Dựa trên notebook 03 Raw; giữ batch 64, micro-batch 8, 10 epoch, EMA=0, Structural Features + Syntax-Aware Swap và chọn checkpoint bằng lossaug. Chỉ thay mapping legacy low bằng directed.

Directed dùng toàn bộ relation ID cho parent/features và swap role (không chia R // 2). Self-loop không phải parent; root dùng sentinel R. Giữ nguyên graph dependent→head, encoder và message passing. Cache hai chiều hoặc có cycle/multiple-parent bị từ chối.

Train/validation: GPT-3.5 Mixed. Test duy nhất: HC3 0% (1.000 mẫu), threshold 0.5. SEED mặc định 2026; có thể đổi thành 2022/2024. Không nạp checkpoint legacy với mapping mới: phải train lại.

Notebook tự patch bản clone Kaggle và đóng gói source đã patch trong ZIP; không sửa source của workspace. Giữ bản notebook 03 để chạy paired baseline cùng seed/batch/môi trường.


In [ ]:
# Kaggle Python 3.13 needs current binary wheels. Do not pin transformers==4.44.2.
!pip install -q --upgrade torch-geometric transformers spacy scikit-learn tqdm
!python -m spacy download en_core_web_sm

import torch
import torch_geometric
import transformers
import spacy
print({
    "torch": torch.__version__,
    "torch_geometric": torch_geometric.__version__,
    "transformers": transformers.__version__,
    "spacy": spacy.__version__,
})


In [ ]:
from pathlib import Path
import json
import shutil
import subprocess
import sys

WORK = Path("/kaggle/working/group_v3_structmap_raw_gpt35_to_hc3_bs64")
REPO = WORK / "PRDetect-Research"
BRANCH = "version3"
SEED = 2026
BATCH_SIZE = 64
MICRO_BATCH_SIZE = 8
EPOCHS = 10
STRUCT_MODE = "directed"
TEST_SPLIT = "hc3_test"
MODEL_DIR = WORK / "model"
EVAL_DIR = WORK / "evaluation"
if REPO.exists():
    shutil.rmtree(REPO)
WORK.mkdir(parents=True, exist_ok=True)
subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH,
                "https://github.com/tphan1420/PRDetect-Research.git", str(REPO)], check=True)
MODEL_DIR.mkdir(exist_ok=True)
EVAL_DIR.mkdir(exist_ok=True)
assert (REPO / "train_v3.py").is_file() and (REPO / "evaluate_v3_quang.py").is_file()
assert (REPO / "data/original_text/benchmarks/hc3/hc3_test.json").is_file()
print({"commit": subprocess.check_output(["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True).strip(),
       "variant": "baseline", "seed": SEED, "effective_batch_size": BATCH_SIZE,
       "micro_batch_size": MICRO_BATCH_SIZE, "test": TEST_SPLIT})


In [ ]:
# Opt-in corrected structural mapping; patch only this notebook's Kaggle clone.
# Legacy low/high behavior and RGCN weights/message passing are unchanged.
def replace_once(text, old, new):
    if text.count(old) != 1:
        raise RuntimeError("Upstream source changed; refusing ambiguous patch: " + old[:100])
    return text.replace(old, new, 1)

model_path = REPO / "model/QuangV3.py"
model_source = model_path.read_text(encoding="utf-8")
model_source = replace_once(
    model_source, '    half = R // 2\n    if mode == "low":',
    '    if mode == "directed":\n        return (et >= 0) & (et < R)\n    half = R // 2\n    if mode == "low":')
model_source = replace_once(
    model_source, '    par[src[m]] = dst[m]\n    return par, int(m.sum()), np.unique(src[m]).size',
    '''    if mode == "directed":
        if np.unique(src[m]).size != int(m.sum()):
            raise ValueError("Directed mapping requires one non-self parent per node")
        if ((et < 0) | (et >= R)).any():
            raise ValueError("Relation ID outside training vocabulary")
    par[src[m]] = dst[m]
    if mode == "directed":
        # Linear-time cycle detection on a parent-pointer forest.
        visited = np.zeros(n, dtype=np.int8)
        for start in range(n):
            if visited[start]:
                continue
            trail = []
            node = start
            while node >= 0 and visited[node] == 0:
                visited[node] = 1
                trail.append(node)
                node = par[node]
            if node >= 0 and visited[node] == 1:
                raise ValueError("Dependency parent cycle")
            visited[trail] = 2
    return par, int(m.sum()), np.unique(src[m]).size''')
model_source = replace_once(
    model_source, '        node_rel[src[m]] = r[m]',
    '''        if self.mode == "directed":
            node_rel = torch.full((N,), self.R, device=dev, dtype=torch.long)
            m = (src != dst)
            r = rel
        node_rel[src[m]] = r[m]''')
compile(model_source, str(model_path), "exec")
model_path.write_text(model_source, encoding="utf-8")

for filename in ("train_v3.py", "evaluate_v3_quang.py"):
    path = REPO / filename
    source = path.read_text(encoding="utf-8")
    old = 'choices=("low", "high")'
    if old not in source:
        raise RuntimeError("Expected legacy structural mode choices in " + filename)
    source = source.replace(old, 'choices=("low", "high", "directed")')
    if filename == "evaluate_v3_quang.py":
        # Correct report provenance for the GPT-3.5 training domain.
        source = replace_once(source, '"train": "hc3"', '"train": config["train_domain"]')
    compile(source, str(path), "exec")
    path.write_text(source, encoding="utf-8")
print("Installed structural mapping: directed (features + swap); legacy modes preserved")


In [ ]:
# Small invariant checks before building expensive embeddings or training.
subprocess.run([sys.executable, "-c", """
import numpy as np
import torch
from model.QuangV3 import parent_array, struct_features, Net
src = np.array([0, 1, 2, 3, 4, 5])
dst = np.array([1, 2, 3, 4, 5, 5])
rel = np.array([12, 25, 34, 42, 50, 0])
par, count, unique = parent_array(6, src, dst, rel, 51, 'directed')
assert par.tolist() == [1, 2, 3, 4, 5, -1] and count == unique == 5
st, _ = struct_features(6, src, dst, rel, 51, 'directed')
assert st[:, 0].tolist() == [5, 4, 3, 2, 1, 0]
legacy, _, _ = parent_array(6, src, dst, rel, 51, 'low')
assert legacy.tolist() == [1, -1, -1, -1, -1, -1]
for s, d in [(np.array([0, 0]), np.array([1, 2])),
             (np.array([0, 1]), np.array([1, 0]))]:
    try:
        parent_array(3, s, d, np.array([34, 42]), 51, 'directed')
    except ValueError:
        pass
    else:
        raise AssertionError('Invalid parent graph accepted')
model = Net(51, in_dim=2, d=4, nb=2, L=1, mode='directed', aug_max=1.0)
x = torch.arange(12).reshape(6, 2).float()
edges = torch.tensor([0, 1, 2, 3, 4, 5])
heads = torch.tensor([5, 5, 5, 5, 5, 5])
roles = torch.tensor([25, 25, 34, 34, 50, 0])
groups = [{0, 1}, {0, 1}, {2, 3}, {2, 3}, {4}, {5}]
torch.manual_seed(2026)
for _ in range(50):
    out = model._augment(x, edges, heads, roles, torch.zeros(6, dtype=torch.long), 1, 'swap', None)
    for i, row in enumerate(out):
        assert int(row[0].item() / 2) in groups[i], 'Cross-role swap'
print('PASS: all relation IDs, depth, roots, cycles, duplicate parents, legacy mode, swap roles')
"""], cwd=REPO, check=True)


In [ ]:
# The test graph is deliberately not built until after V3 training.
subprocess.run([sys.executable, "building_graph.py", "--file", "gpt3.5_mixed_train_split", "gpt3.5_mixed_val_split"], cwd=REPO, check=True)


In [ ]:
subprocess.run(
    [sys.executable, "train_v3.py", "--variants", "baseline", "--seeds", str(SEED),
     "--graph-dir", str(REPO / "graph_data"), "--output-dir", str(MODEL_DIR),
     "--train-file", "gpt3.5_mixed_train_split", "--val-file", "gpt3.5_mixed_val_split", "--struct-mode", STRUCT_MODE,
     "--batch-size", str(BATCH_SIZE), "--micro-batch-size", str(MICRO_BATCH_SIZE),
     "--val-batch-size", "64", "--epochs", str(EPOCHS), "--patience", "10", "--ema", "0", "--device", "cuda"],
    cwd=REPO,
    check=True,
)
CHECKPOINT = MODEL_DIR / f"v3_baseline_seed{SEED}.pt"
TRAIN_METADATA = CHECKPOINT.with_suffix(".train.json")
assert CHECKPOINT.is_file() and TRAIN_METADATA.is_file()

# --ema 0 means the selected/evaluated state is exactly the raw model state.
checkpoint_payload = torch.load(CHECKPOINT, map_location="cpu", weights_only=True)
assert checkpoint_payload["args"]["ema"] == 0
assert checkpoint_payload["struct_mode"] == STRUCT_MODE
assert checkpoint_payload["state"].keys() == checkpoint_payload["raw_state"].keys()
assert all(torch.equal(checkpoint_payload["state"][key], checkpoint_payload["raw_state"][key])
           for key in checkpoint_payload["state"])
del checkpoint_payload

In [ ]:
subprocess.run([sys.executable, "building_graph.py", "--file", TEST_SPLIT], cwd=REPO, check=True)
GRAPH = REPO / "graph_data" / f"{TEST_SPLIT}.pkl"
assert GRAPH.is_file()

# Use the repository evaluator's strict loader and its internal one-split worker.
from types import SimpleNamespace
import importlib.util
spec = importlib.util.spec_from_file_location("v3_eval", REPO / "evaluate_v3_quang.py")
v3_eval = importlib.util.module_from_spec(spec)
spec.loader.exec_module(v3_eval)
args = SimpleNamespace(variant="baseline", train_domain="gpt35", checkpoint_dir=str(MODEL_DIR), seeds=[SEED],
                       batch_size=BATCH_SIZE, struct_mode=STRUCT_MODE, device="cuda", graph_budget_root=str(WORK))
config = v3_eval.make_config(args)
config["weight_source"] = "raw"
config["structural_mapping"] = "directed_all_dependency_relations"
CONFIG = EVAL_DIR / "config.json"
v3_eval.write_json(CONFIG, config)
expected_samples = sum(1 for line in (REPO / "data/original_text/benchmarks/hc3/hc3_test.json").open(encoding="utf-8") if line.strip())
assert expected_samples == 1000
REPORT = EVAL_DIR / "hc3_0pct_raw_report.json"
subprocess.run([sys.executable, "evaluate_v3_quang.py", "_test", "--config", str(CONFIG), "--seed", str(SEED),
                "--graph", str(GRAPH), "--expected-samples", str(expected_samples),
                "--signature", "structmap_raw_hc3_0pct_after_gpt35_training", "--report", str(REPORT)], cwd=REPO, check=True)
assert REPORT.is_file()

In [ ]:
import zipfile
report = json.loads(REPORT.read_text(encoding="utf-8"))
archive = WORK / f"v3_structmap_raw_gpt35_to_hc3_0pct_seed{SEED}_batch{BATCH_SIZE}_results.zip"
with zipfile.ZipFile(archive, "w", zipfile.ZIP_DEFLATED) as out:
    for path in (CHECKPOINT, TRAIN_METADATA, CONFIG, REPORT, REPO / "model/QuangV3.py",
                 REPO / "train_v3.py", REPO / "evaluate_v3_quang.py"):
        # Keep patched sources to reproduce directed checkpoint loading.
        out.write(path, path.name)
print(json.dumps({key: report[key] for key in ("samples", "accuracy", "auc", "f1", "recall_ai", "recall_human", "threshold", "validation_threshold")}, indent=2))
print("ZIP:", archive)